# Agentic RAG 与框架对比

## Multi-Agent Frameworks — AutoGen 与 CrewAI 实战指南

LangGraph 选型 · AutoGen / CrewAI / LangGraph 对比 · Agentic RAG


> ⚠️ **前置**：请先运行 [`03_环境初始化与MAS基础.ipynb`](03_环境初始化与MAS基础.ipynb) 完成依赖安装与环境变量配置。


## 📖 讲义
### 课程资料

3. 实践示例（Practical Examples）
下面的示例以教学为主，展示三个层次的实现思路：AutoGen 风格的快速原型、CrewAI 风格的协调器/队列模式和一个混合 RAG 多 Agent 场景。代码以伪代码/简化 Python 为主，重点展示模式与数据流，方便同学改写为真实框架 API。

说明：真实使用请参考 AutoGen / CrewAI 官方文档与 SDK。

示例 A — AutoGen 风格：快速定义三个角色的协作（研究员 → 写作者 → 审校者）
# pseudo_autogen.py (伪代码)
# 定义 agents
researcher = Agent(name="Researcher", capability="search_and_extract")
writer = Agent(name="Writer", capability="compose_summary")
editor = Agent(name="Editor", capability="fact_check_and_edit")

# Orchestrator 创建计划
def orchestrate(topic):
    # step1: researcher 查资料
    docs = researcher.run({"task":"search", "query":topic})
    # step2: writer 用 docs 写初稿
    draft = writer.run({"task":"write", "context": docs})
    # step3: editor 校验并返回 final
    final = editor.run({"task":"verify_and_edit", "draft": draft, "context": docs})
    return final
教学点：AutoGen 便于通过少量代码声明 Agent 并运行“角色扮演”式流程；同学可在本地先用小模型/模拟返回替代实际检索。

示例 B — CrewAI / 编队（Crew）模式：Orchestrator + Worker 池（消息队列）
# orchestrator_queue.py (伪)
# Orchestrator 将任务拆成子任务并写入队列
job_id = create_job(topic)
push_queue("retriever", {"job_id": job_id, "topic": topic})
# Retriever Worker 读取，检索并推送下一队列
# Reranker Worker 处理，并推送 generator queue
# Generator Worker 生成并把结果写回 result db
工程点：

每个队列都能水平扩容 Worker；

job_id 用于聚合结果；

监控队列长度、任务成功率与 worker 健康。

示例 C — Agentic RAG 的多 Agent 协作（检索/生成/验证/调度）
# agentic_rag_multi.py (伪)
# 1. Orchestrator: 接收 query，派发检索
retrieval_job = dispatch("retriever", {"query": q, "top_k": 50})
candidates = wait_for_result(retrieval_job)

# 2. Reranker Agent: 精排 top_k -> top_5
top5 = call_agent("reranker", {"query": q, "candidates": candidates})

# 3. Generator Agent: 用 top5 生成答案草稿
draft = call_agent("generator", {"query": q, "context": top5})

# 4. Verifier Agent: NLI 检验关键断言并给出 pass/fail
verify = call_agent("verifier", {"draft": draft, "context": top5})

# 5. Orchestrator: 若 verify.pass -> return draft; else -> HITL or retry
if verify["pass"]:
    return draft
else:
    send_to_human_review(draft, verify)
实践点：该流程体现分层/协作/监督三要素：各 Agent 专注任务；Orchestrator 聚合；Verifier 做质量门。

---

### 幻灯片

# Multi-Agent Frameworks
## AutoGen 与 CrewAI 实战指南

课程学习资料 · 面向零基础 AI Engineer 训练营

含 LangGraph 对比与选型 · 120 分钟 · 88 页

---

## 四、LangGraph：为什么要拿来比较

- LangGraph 常被用来做显式、可恢复的 agent 工作流
- 它的重心是 graph、state 和 routing
- 很适合帮助我们理解“状态驱动”的协作方式
- 它不是 AutoGen 或 CrewAI 的简单替代品
- 三者更像站在不同抽象层上的选择

> 讲师提示：这里把对比重心放在“心智模型差异”，不要做站队。

---

## LangGraph 的 Graph / Node / Edge / State 心智模型

- Graph 是整个工作流骨架
- Node 是执行单元，可以是 LLM、工具或子流程
- Edge 决定下一步去哪里
- State 是流程共享、持续演化的数据
- 显式状态让控制流更容易被看见和调试

一句话：

- 消息驱动关注“说了什么”，状态驱动关注“现在到哪一步了”

---

## LangGraph 的 Router、分支与 Stateful Workflow

- 条件边可以根据结果做路由
- 可以混合确定性逻辑和 LLM 决策
- 很适合 plan -> act -> verify 这类循环
- 也适合把复杂流程拆成多个可复用节点或子图
- 当你想“看见每一条分支”时，graph 模型非常顺手

适合的流程特征：

- 分支多
- 状态重
- 需要检查点

---

## LangGraph 的 Checkpoint、持久化、HITL 与长流程

- 官方文档强调 persistence 和 checkpointing
- 可以在任意中间状态暂停和恢复
- interrupts 让人类介入点更自然
- 对分钟级、小时级流程更友好
- 当审批、修订、恢复很重要时，优势会很明显

课堂要点：

- “可恢复”本身就是重要产品能力

---

## LangGraph 的优势、局限与典型场景

- 优势：控制流显式、状态可追踪、恢复能力强
- 优势：适合状态图工作流、审批流、分支复杂流程
- 局限：概念门槛更高，设计前置成本更大
- 局限：对零基础同学来说，不如角色协作那样直观
- 典型场景：带审批、持久化、复杂分支的 agent 系统

结论：

- 当“状态”和“恢复”比“对话感”更重要时，可以优先想它

---

## 三框架总览表

| 维度 | AutoGen | CrewAI | LangGraph |
| --- | --- | --- | --- |
| 核心重心 | 对话协作 | 任务编排 | 状态图工作流 |
| 主要抽象 | Agent / Team / Message | Agent / Task / Crew | Node / Edge / State |
| 状态载体 | 消息与团队上下文 | 任务执行上下文 | 显式共享状态 |
| 初学体验 | 最容易起步 | 中等 | 概念更重 |
| 更适合 | 原型、教学 | 自动化流程 | 复杂分支、持久化 |

---

## 架构维度对比

| 问题 | AutoGen | CrewAI | LangGraph |
| --- | --- | --- | --- |
| 流程靠什么推进 | 多轮消息 | 任务与 process | 节点与边 |
| 中心关注点 | 谁来对话 | 谁来做任务 | 当前状态是什么 |
| 强项 | 角色协作感 | 任务交付感 | 控制流可视性 |
| 容易失控处 | 对话轮次 | 任务边界 | 图设计复杂度 |

课堂结论：

- 消息、任务、状态，是三种不同的组织世界观

---

## 工程维度对比

| 维度 | AutoGen | CrewAI | LangGraph |
| --- | --- | --- | --- |
| 可控性 | 中 | 中高 | 高 |
| 可观测性 | 对话/trace 友好 | 运行/任务友好 | 状态/路径友好 |
| 长流程恢复 | 需额外设计 | 取决于外围流程 | 原生心智更强 |
| 学习曲线 | 低 | 中 | 中高 |
| 适合课堂起步 | 很适合 | 适合进阶 | 适合做对比理解 |

---

## 选型决策页

- 目标是教学原型、角色协作演示：优先 **AutoGen**
- 目标是任务编排、交付流程、自动化执行：优先 **CrewAI**
- 目标是状态图工作流、审批流、可恢复长流程：优先 **LangGraph**
- 如果不确定：先用最简单的抽象，不要一开始就堆满多 Agent
- 常见路线：先用 AutoGen 验证价值，再用 CrewAI 或 LangGraph 产品化

> 讲师提示：先选抽象模型，再选框架名字。

---

## 六、实战场景

- 下面三个示例都围绕同一个思路展开
- 先拆分职责，再组织数据流，再加监督与回退
- 代码是伪代码，重点是模式，不是 API 细节
- 同一个场景可以用不同框架建模，但重心不同

> 📖 完整讲义已嵌入本节（原 `课程资料.md` / `Marp 幻灯片.md` 已删除）


---

## Part 4 — Agentic RAG 多 Agent

> 将 RAG（检索增强生成）拆解为多个专职 Agent，每个 Agent 只负责一个环节，整体由 LangGraph `StateGraph` 编排。

### 架构

```
User Query
    │
    ▼
Orchestrator ──► Retriever ──► Reranker ──► Generator ──► Verifier
                                                               │
                                              pass ◄──────────┤
                                              fail ──► HITL Review
```

In [ ]:
from typing import TypedDict, List, Optional
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

# 通过 OpenRouter 初始化 LangChain 客户端
llm_sonnet = ChatOpenAI(
    model=MODEL_SONNET,
    openai_api_key=OPENROUTER_API_KEY,
    openai_api_base=OPENROUTER_BASE_URL,
    max_tokens=1024,
)

llm_haiku = ChatOpenAI(
    model=MODEL_HAIKU,
    openai_api_key=OPENROUTER_API_KEY,
    openai_api_base=OPENROUTER_BASE_URL,
    max_tokens=512,
)

# 共享状态
class AgenticRAGState(TypedDict):
    query: str
    candidates: List[str]
    top_docs: List[str]
    draft_answer: str
    verify_result: dict
    final_answer: Optional[str]
    needs_review: bool

print('✅ LangGraph + OpenRouter 客户端初始化完成')

In [ ]:
# 模拟知识库
KNOWLEDGE_BASE = [
    'AutoGen 是微软开发的多 Agent 框架，支持 GraphFlow、SelectorGroupChat 等多种协作模式。',
    'CrewAI 提供高层级抽象，通过 Agent、Task、Crew 三个核心概念实现多 Agent 协作。',
    'LangGraph 使用有向图表达 Agent 工作流，适合需要循环、分支和状态管理的复杂流程。',
    '多 Agent 系统的常见通信模式包括：同步 RPC、异步消息队列、共享黑板（Blackboard）。',
    'Agentic RAG 将检索过程拆解为：粗检索、精排、生成、验证四个阶段，提升答案质量。',
    'Agent 可观测性工具包括：LangSmith、Arize、Weights & Biases，支持 Trace 记录和性能分析。',
    '多 Agent 系统的常见陷阱：无限循环、过度中心化 Orchestrator、共享状态竞态条件。',
    'Quality Gate（质量门）在关键节点插入验证 Agent，确保输出满足质量标准后才流向下一步。',
    'HITL（人在回路）通过将低置信度结果路由到人工审查，平衡自动化与准确性。',
    'OpenRouter 提供统一的 OpenAI 兼容接口，支持路由到 Claude、GPT、Gemini 等多种模型。',
]


def retriever_node(state: AgenticRAGState) -> dict:
    query = state['query']
    keywords = query.lower().split()
    scored = [(sum(1 for kw in keywords if kw in doc.lower()), doc) for doc in KNOWLEDGE_BASE]
    candidates = [doc for score, doc in sorted(scored, reverse=True) if score > 0][:5]
    if not candidates:
        candidates = KNOWLEDGE_BASE[:3]
    print(f'[Retriever] 检索到 {len(candidates)} 条候选')
    return {'candidates': candidates}


def reranker_node(state: AgenticRAGState) -> dict:
    query, candidates = state['query'], state['candidates']
    numbered = '\n'.join(f'{i+1}. {doc}' for i, doc in enumerate(candidates))
    response = llm_haiku.invoke([
        SystemMessage(content='选出与查询最相关的 3 条文档，只输出编号（逗号分隔，如：1,3,5）。'),
        HumanMessage(content=f'查询：{query}\n\n候选：\n{numbered}\n\n最相关编号：'),
    ])
    try:
        indices = [int(x.strip()) - 1 for x in response.content.split(',') if x.strip().isdigit()]
        top_docs = [candidates[i] for i in indices if 0 <= i < len(candidates)]
    except Exception:
        top_docs = candidates[:3]
    print(f'[Reranker] 精排保留 {len(top_docs)} 条')
    return {'top_docs': top_docs}


def generator_node(state: AgenticRAGState) -> dict:
    context = '\n'.join(f'• {doc}' for doc in state['top_docs'])
    response = llm_sonnet.invoke([
        SystemMessage(content='仅根据参考文档回答问题，200 字以内。'),
        HumanMessage(content=f'参考文档：\n{context}\n\n问题：{state["query"]}'),
    ])
    print(f'[Generator] 生成答案（{len(response.content)} 字）')
    return {'draft_answer': response.content}


def verifier_node(state: AgenticRAGState) -> dict:
    context = '\n'.join(f'• {doc}' for doc in state['top_docs'])
    response = llm_haiku.invoke([
        SystemMessage(content=(
            '检验答案与参考文档的一致性。'
            '输出 JSON：{"pass": true/false, "confidence": 0.0-1.0, "issues": [...]}'
        )),
        HumanMessage(content=f'参考：\n{context}\n\n答案：\n{state["draft_answer"]}'),
    ])
    import json, re
    try:
        m = re.search(r'\{.*\}', response.content, re.DOTALL)
        result = json.loads(m.group()) if m else {'pass': True, 'confidence': 0.7, 'issues': []}
    except Exception:
        result = {'pass': True, 'confidence': 0.7, 'issues': []}
    status = '✅ 通过' if result.get('pass') else '❌ 未通过'
    print(f'[Verifier] {status} (置信度: {result.get("confidence", 0):.2f})')
    return {'verify_result': result}


def orchestrator_route(state: AgenticRAGState) -> str:
    v = state.get('verify_result', {})
    return 'accept' if (v.get('pass', True) and v.get('confidence', 0.7) >= 0.6) else 'review'


def accept_node(state: AgenticRAGState) -> dict:
    print('[Orchestrator] ✅ 答案接受')
    return {'final_answer': state['draft_answer'], 'needs_review': False}


def review_node(state: AgenticRAGState) -> dict:
    print(f'[Orchestrator] ⚠️  需要人工审查：{state.get("verify_result", {}).get("issues", [])}')
    return {'needs_review': True, 'final_answer': None}


print('✅ Agent 节点函数定义完成')

In [ ]:
# 构建图
rag_builder = StateGraph(AgenticRAGState)
rag_builder.add_node('retriever', retriever_node)
rag_builder.add_node('reranker', reranker_node)
rag_builder.add_node('generator', generator_node)
rag_builder.add_node('verifier', verifier_node)
rag_builder.add_node('accept', accept_node)
rag_builder.add_node('review', review_node)
rag_builder.add_edge(START, 'retriever')
rag_builder.add_edge('retriever', 'reranker')
rag_builder.add_edge('reranker', 'generator')
rag_builder.add_edge('generator', 'verifier')
rag_builder.add_conditional_edges('verifier', orchestrator_route, {'accept': 'accept', 'review': 'review'})
rag_builder.add_edge('accept', END)
rag_builder.add_edge('review', END)

agentic_rag = rag_builder.compile()

# 可视化
try:
    from IPython.display import Image, display
    display(Image(agentic_rag.get_graph().draw_mermaid_png()))
except Exception:
    print('流程：START → retriever → reranker → generator → verifier → [accept|review] → END')

print('✅ Agentic RAG 图构建完成')

In [ ]:
# 运行 Agentic RAG
test_query = 'AutoGen 和 CrewAI 有什么区别，各自适合什么场景？'
print(f'❓ 查询：{test_query}\n' + '─' * 60)

final_state = agentic_rag.invoke({
    'query': test_query,
    'candidates': [],
    'top_docs': [],
    'draft_answer': '',
    'verify_result': {},
    'final_answer': None,
    'needs_review': False,
})

print('─' * 60)
if final_state['final_answer']:
    print(f'\n✅ 最终答案：\n{final_state["final_answer"]}')
else:
    print(f'\n⚠️  需要人工审查')
    print(f'    草稿：{final_state["draft_answer"][:200]}...')